# aw_06_b3 — Stage B3: P1 general DPO on the Phase-1 SFT champion (Track B, §5.1/§6)

**Champion selection record (2026-08-10, protocol §6 three-stage on the V2 probes):**
transfer proxy (probe eval-ID pass_rate) 0.2567 (b1v2) vs 0.2500 (b2v2), p=0.88 → tie;
rule-OOD 0.1933 vs 0.2100, p=0.61 → tie; GPU-hours tie-break (B1 requires no RS
generation pass) → **champion = B1v2**
(`20260807-225109--b1-general-sft-v2--s42--e6e83b`, m97j/aw-runs-b1).

Cell order: `a_b3_data` (mine exact-answer pairs from the champion policy) →
`b_b3_train` (DPO, lineage-verified) → `c_b3_eval` (P1 held-out) →
`d_b3_probe` → `e_b3_probe_eval` → `x09e_run_audit` → `f_b3_analysis`.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title fetch champion — materialize the b1v2 final adapter + lineage sha
B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b1v2_dir, b1v2_sha)


In [ ]:
# @title a_b3_data — mine exact-answer preference pairs from the champion (GPU)
# Same mining engine/schema as A2 (generation.pair_mining); verifier is the
# frozen ExactAnswerVerifier against gold answers in the P1 record metadata.
# Report pair_yield + decision_counts from the manifest in the checklist.
!python scripts/mine_p1_pairs.py \
  --config configs/experiments/b3_general_dpo.yaml \
  --adapter-dir {b1v2_dir} \
  --input data/p1/p1_general_sft.jsonl \
  --num-candidates 8 --temperature 0.8 --max-new-tokens 768 --batch-size 64 \
  --output data/p1/p1_general_preference.jsonl \
  --hf-sync-repo m97j/axiom-general-posttrain --hf-path-in-repo p1/pref-v1


In [ ]:
# @title b_b3_train — DPO from the b1v2 parent (lineage-verified)
!python scripts/run_experiment.py \
  --config configs/experiments/b3_general_dpo.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override data.source.local_path=data/p1/p1_general_preference.jsonl \
  --hf-sync-repo m97j/aw-runs-b3


In [ ]:
# @title c_b3_eval — P1 held-out accuracy (vs b1v2 0.844) + per-row dump
B3_RUN_ID = ""  # <- from b_b3_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b3 --run-id {B3_RUN_ID}
b3_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_p1_eval.py \
  --config configs/experiments/b3_general_dpo.yaml \
  --adapter-dir {b3_dir} --label b3-dpo \
  --output runs/p1_eval_b3.json \
  --dump-predictions runs/p1_eval_b3_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b3

!python scripts/x09_termination_audit.py \
  --p1-predictions runs/p1_eval_b3_predictions.jsonl --out runs/x09_drift_audit_b3.json


In [ ]:
# @title d_b3_probe — frozen transfer probe (200 steps) from the B3 policy
import json
b3_sha = json.load(open(f"runs/{B3_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b3_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b3 \
  --override lineage.parent_adapter.sha256={b3_sha} \
  --override experiment_name=probe-playworld-sft-b3 \
  --hf-sync-repo m97j/aw-runs-b3-probe


In [ ]:
# @title e_b3_probe_eval — B3 probe adapter on the frozen suites
B3_PROBE_RUN = ""  # <- from d_b3_probe

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b3-probe --run-id {B3_PROBE_RUN}
b3p_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b3p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b3-probe


In [ ]:
# @title x09e_run_audit — termination regression check on the B3 probe eval (CPU)
B3_PROBE_EVAL = ""  # <- eval run id from e_b3_probe_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b3-probe --run-id {B3_PROBE_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B3_PROBE_EVAL} --out runs/x09_run_audit_b3.json


In [ ]:
# @title f_b3_analysis — B3 probe vs b1v2 probe (primary: does P1 DPO help transfer?)
B1V2_PROBE_EVAL = "20260810-063029--eval-playworld--s42--a7a47a"

!python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1V2_PROBE_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B3_PROBE_EVAL} --label-a b3-probe \
  --run-b runs/{B1V2_PROBE_EVAL} --label-b b1v2-probe \
  --output runs/{B3_PROBE_EVAL}/analysis_b3_vs_b1v2_probe.json --hf-sync-repo m97j/aw-runs-b3-probe


## Stage checklist
- [ ] `p1_preference_manifest.json`: pairs_accepted, pair_yield, decision_counts recorded
- [ ] b_b3_train lineage verified (parent sha = b1v2 output_adapter_sha256)
- [ ] P1 held-out: b3 accuracy vs b1v2 0.844 (retention: no drop > 3 pts) + x09d drift ≈ 0
- [ ] x09e: truncation_rate ≈ 0, runaway_rate ≈ 0 (stop behavior survives DPO)
- [ ] f_b3_analysis: eval-ID / OOD deltas vs b1v2-probe with CIs recorded
- [ ] OUTSTANDING (carried from aw_05): base-model P1 held-out accuracy + RS manifest
      acceptance_rate — needed to finalize the §6 Stage-1 retention record
